# 05. Paper Handoff, Figures, and Reproducibility Map

This final notebook maps the analysis notebooks to release-safe aggregate
tables and figures. It does not select or regenerate a manuscript PDF; the
current blind-review manuscript should be verified separately from this
public artifact package.


<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.

**Public-package behavior:** If authorized processed data are unavailable, this notebook skips private-data computation cells without raising an error. This silent skip is expected and does not reproduce manuscript estimates. For a fully runnable bundled-data example, open `../01_metric_walkthrough_synthetic.ipynb` or run `python scripts/run_public_synthetic_pipeline.py` from the repository root.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:handoff_guide -->

## Reader guide: what this handoff notebook checks

This notebook connects regenerated analysis artifacts back to the manuscript. It is the final sanity-check layer before paper submission or repository release.

Use it to answer:

1. Which tables support each RQ?
2. Which figures are manuscript figures versus supplementary/release figures?
3. Which files are safe to commit, and which private row-level artifacts should stay out of GitHub?

The goal is traceability from claim -> artifact -> code path, without exposing raw operational logs.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "05_paper_handoff_and_artifacts"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PROCESSED.exists()}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PROCESSED.exists():
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`."
    ))


In [ ]:
figure_files = [
    "pec_audit_framework.png",
    "fig15_surface_traceability_liftplot_v37.png",
    "fig16_diversity_audit_v46.png",
    "fig17_bounded_audit_signal_native_plotly_v21.png",
    "release_popularity_baseline.png",
    "release_surface_lift.png",
]
fig_table = pd.DataFrame({
    "figure_file": figure_files,
    "exists": [(FIGURES / f).exists() for f in figure_files],
    "path": [str(FIGURES / f) for f in figure_files],
})
display(fig_table)


In [ ]:
# Display a compact artifact inventory.
def display_folder(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

reports = []
for folder in [TABLES, FIGURES]:
    if folder.exists():
        for path in sorted(folder.glob("*")):
            if path.is_file():
                reports.append({
                    "folder": display_folder(folder),
                    "file": path.name,
                    "size_kb": round(path.stat().st_size / 1024, 1),
                })
inventory = pd.DataFrame(reports)
display(inventory.groupby("folder").agg(files=("file", "count"), total_kb=("size_kb", "sum")))
display(inventory.sort_values(["folder", "file"]).head(80))


## Recommended reading order

1. `00_data_eda_and_schema.ipynb`: understand the processed tables.
2. `01_preprocessing_and_joining.ipynb`: rebuild preference-consumption and exposure-consumption joins.
3. `02_pec_metrics_and_statistics.ipynb`: compute PEC metrics and count-matched diversity checks.
4. `03_weighted_preferences_and_surface_paths.ipynb`: inspect weighted profile-state and app surfaces.
5. `04_modeling_rq4_from_features.ipynb`: run diagnostic models and robustness checks.
6. `05_paper_handoff_and_artifacts.ipynb`: connect results to the selected PDF/figures.

## Existing notebooks vs this folder

The existing notebooks in `notebooks/` are useful for reviewing final
outputs. This `repro_deep/` folder is for learning, debugging, and
reproducing the calculations step by step.


In [ ]:
# Optional release-safe commands. Run these in a terminal from the project
# root. They are shown here rather than executed automatically to avoid
# overwriting chosen paper artifacts.
commands = [
    "python scripts/run_public_synthetic_pipeline.py",
    "python scripts/check_release_artifacts.py",
    "python scripts/make_release_figures.py",
]
for cmd in commands:
    print(cmd)


## Claim-to-artifact map

This final notebook is not primarily for new computation. It checks the link between manuscript claims and their supporting artifacts. Before submission, each claim should be traceable to a table, figure, or code path.


In [ ]:
claim_to_artifact = pd.DataFrame([
    {
        "paper component": "Figure 1 / Framework",
        "supporting artifact": "reports/figures/pec_audit_framework.png",
        "what to verify": "P/E/C layers, S conditioning context, RQ1-RQ4 labels, observability boundaries match manuscript text",
    },
    {
        "paper component": "RQ1 preference-consumption alignment",
        "supporting artifact": "rq1_preference_consumption_gap_by_user.csv; preference_weighted_features.csv; Figure 2",
        "what to verify": "698 users, empty stated-category rule, Jaccard/cosine/top-category match values",
    },
    {
        "paper component": "RQ2 exposure-consumption traceability",
        "supporting artifact": "rq2_click_exposure_rows.csv; optimization_topk_exposure.csv; Figure 3",
        "what to verify": "same-day comparable-click denominator, candidate-pool baseline, top-k/gap, surface wording",
    },
    {
        "paper component": "RQ3 diversity/concentration mismatch",
        "supporting artifact": "rq3_diversity_by_user.csv; count-matched diversity tables; Figure 4",
        "what to verify": "raw vs count-matched distinction, entropy gap, HHI/top-share/JS values",
    },
    {
        "paper component": "RQ4 structured divergence",
        "supporting artifact": "rq4_divergence_model_features.csv; model CV/stress tables; Figure 5",
        "what to verify": "AUC values, dense-history/future-window checks, label-proximity sensitivities",
    },
    {
        "paper component": "Checklist and reproducibility promises",
        "supporting artifact": "release-safe notebooks, scripts, aggregate tables, synthetic schema, README/license files",
        "what to verify": "claims match what is actually submitted or promised for post-acceptance release",
    },
])
display(claim_to_artifact)


## Release boundary: what can and cannot be pushed or submitted

Internal manuscript verification can use row-level processed data, but those records must not be released through GitHub or supplementary materials. Public artifacts are limited to aggregate tables, figure-generation code, synthetic schemas, and release-safe notebooks that do not expose private user traces.


In [ ]:
release_boundary = pd.DataFrame([
    {
        "artifact type": "private processed row-level data",
        "examples": "user_profiles_clean.csv, behavior_events_dedup.csv, click_events_enriched.csv with row-level IDs",
        "default action": "do not commit / do not public-submit unless explicitly authorized and de-identified",
    },
    {
        "artifact type": "aggregate result tables",
        "examples": "RQ summaries, model CV summaries, figure source tables without row-level traces",
        "default action": "can be included if checked for anonymity and license wording",
    },
    {
        "artifact type": "release-safe notebooks and scripts",
        "examples": "notebooks using public aggregates/synthetic schemas or authorized local-path config",
        "default action": "safe to publish after metadata/path/user-name scrub",
    },
    {
        "artifact type": "manuscript/Overleaf source",
        "examples": "main.tex, figures, bibliography, and style files kept outside this public artifact repository",
        "default action": "submit through the conference system; keep blind metadata clean",
    },
])
display(release_boundary)


In [ ]:
# AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
# This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
# It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
_save_names = ['fig_table', 'inventory', 'claim_to_artifact', 'release_boundary']
_saved_paths = []
for _name in _save_names:
    if _name not in globals():
        continue
    _obj = globals()[_name]
    _path = SCRATCH_TABLES / f"{_name}.csv"
    try:
        if isinstance(_obj, pd.DataFrame):
            _obj.to_csv(_path, index=False)
        elif isinstance(_obj, pd.Series):
            _obj.to_frame().to_csv(_path)
        elif isinstance(_obj, list):
            pd.DataFrame(_obj).to_csv(_path, index=False)
        elif isinstance(_obj, dict):
            pd.DataFrame([_obj]).to_csv(_path, index=False)
        else:
            continue
        _saved_paths.append(_path)
    except Exception as exc:
        print(f"Could not save {_name}: {exc}")

try:
    import matplotlib.pyplot as _plt
    for _idx, _num in enumerate(_plt.get_fignums(), 1):
        _fig = _plt.figure(_num)
        _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
        _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
        _saved_paths.append(_fig_path)
except Exception as exc:
    print(f"Could not save open matplotlib figures: {exc}")

print("Saved scratch artifacts:")
for _path in _saved_paths:
    print(" -", _path)
